# BiLSTM Performance Report Figures  
**Sensor EM-300 — 24h Temperature & Humidity Forecasting**

notebook นี้ประมวลผลโมเดล BiLSTM บน test set และสร้างรูปคุณภาพสูง (300 DPI)  
สำหรับนำไปใส่รายงาน — ไฟล์ทั้งหมดถูก save ไว้ที่ `reports/`

| Section | เนื้อหา |
|---|---|
| Fig 1 | Overall Metrics (Train/Val/Test) |
| Fig 2 | Per-Horizon RMSE & MAE |
| Fig 3 | Actual vs Predicted Time Series |
| Fig 4 | Scatter Plot & Residuals |
| Fig 5 | Error Heatmap (Hour x Horizon) |
| Fig 6 | Summary Dashboard |
| Fig 7 | Training Loss History |
| Summary | สรุปผล + บทวิเคราะห์สำหรับรายงาน |


---
## 0. Setup & Load

**เซลล์นี้ทำอะไร:**

- Import library ที่จำเป็น: `numpy`, `pandas`, `matplotlib`, `seaborn`
- ตั้งค่า project root แบบ **idempotent** (รันซ้ำได้ปลอดภัย)
- กำหนด path สำคัญ: `DATA_DIR`, `MODEL_DIR`, `OUT_DIR`
- ตั้งค่า matplotlib style สำหรับรูปคุณภาพสูง

**สิ่งที่คาดว่าจะเห็น:**
```
ROOT : .../Predict_temp_hum
OUT  : .../Predict_temp_hum/reports
```


In [4]:
import sys, os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# idempotent root
_here = Path(os.path.abspath('.'))
ROOT  = _here.parent if _here.name == 'notebooks' else _here
os.chdir(ROOT)
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None

DATA_DIR  = ROOT / 'data' / 'processed'
MODEL_DIR = ROOT / 'models'
OUT_DIR   = ROOT / 'reports'
OUT_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 300,
    'font.size': 11, 'axes.titlesize': 12,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
    'savefig.bbox': 'tight', 'savefig.pad_inches': 0.15,
})
%matplotlib inline

COLORS = {
    'temp':  '#c0392b',
    'hum':   '#1a5276',
    'train': '#27ae60',
    'val':   '#f39c12',
    'test':  '#e74c3c',
}
print('ROOT :', ROOT)
print('OUT  :', OUT_DIR)


ROOT : c:\Users\cheew\Desktop\Swift\Project\Predict_temp_hum
OUT  : c:\Users\cheew\Desktop\Swift\Project\Predict_temp_hum\reports


### โหลด Data และ Model

**เซลล์นี้ทำอะไร:**

- **โหลด CSV**: train / val / test จาก `data/processed/`
- **โหลด Model**: `lstm_temp.keras` และ `lstm_hum.keras` จาก `models/`
- **โหลด Scaler**: `lstm_temp_meta.pkl` และ `lstm_hum_meta.pkl` ซึ่งเก็บ `scaler_X`, `scaler_y`, และ config (LOOKBACK, FORECAST)
- **ทำนาย**: รัน model บน train / val / test set

**ผลที่คาดว่าจะเห็น:**
```
train: XXXX rows | val: XXXX rows | test: XXXX rows
temp model loaded | hum model loaded
Evaluating... done
```


In [5]:
def load_csv(name):
    df = pd.read_csv(DATA_DIR / name, index_col=0, parse_dates=True)
    df.index = pd.to_datetime(df.index)
    return df

train_df = load_csv('train.csv')
val_df   = load_csv('val.csv')
test_df  = load_csv('test.csv')
print(f'train: {len(train_df)} rows | val: {len(val_df)} rows | test: {len(test_df)} rows')

import pickle, tensorflow as tf
from src.trainnig.lstm_trainer import add_time_features, make_sequences

def load_lstm_bundle(target, model_dir):
    with open(model_dir / f'lstm_{target}_meta.pkl', 'rb') as f:
        meta = pickle.load(f)
    meta['model'] = tf.keras.models.load_model(str(model_dir / f'lstm_{target}.keras'))
    return meta

bundles = {t: load_lstm_bundle(t, MODEL_DIR) for t in ['temp','hum']}
print('temp model loaded | hum model loaded')

LOOKBACK_STEPS  = bundles['temp']['lookback']
FORECAST_STEPS  = bundles['temp']['n_hours']
STEPS_PER_HOUR  = bundles['temp']['steps_per_hour']
print(f'lookback={LOOKBACK_STEPS}  forecast={FORECAST_STEPS}  steps_per_hour={STEPS_PER_HOUR}')

def evaluate(df, target):
    col_idx = 0 if target == 'temp' else 1
    b   = bundles[target]
    raw = add_time_features(df)                    # (N, 8) numpy
    # scale ให้ตรงกับที่ train: scaler_X ทุก col แล้ว override target ด้วย scaler_y
    Xs  = b['scaler_X'].transform(raw.copy())
    Xs[:, col_idx] = b['scaler_y'].transform(
        raw[:, col_idx].reshape(-1, 1)
    ).flatten()
    X, ys = make_sequences(Xs, col_idx,
                           lookback=LOOKBACK_STEPS,
                           steps_per_hour=STEPS_PER_HOUR,
                           n_hours=FORECAST_STEPS)
    yp_s  = b['model'].predict(X, batch_size=256, verbose=0)
    sy    = b['scaler_y']
    yt    = sy.inverse_transform(ys.reshape(-1,1)).reshape(ys.shape)
    yp    = sy.inverse_transform(yp_s.reshape(-1,1)).reshape(yp_s.shape)
    rmse  = float(np.sqrt(np.mean((yt-yp)**2)))
    mae   = float(np.mean(np.abs(yt-yp)))
    mape  = float(np.mean(np.abs((yt-yp)/(np.abs(yt)+1e-8)))*100)
    r2    = float(1 - np.sum((yt-yp)**2)/np.sum((yt-np.mean(yt))**2))
    h_rmse = [float(np.sqrt(np.mean((yt[:,h]-yp[:,h])**2))) for h in range(FORECAST_STEPS)]
    h_mae  = [float(np.mean(np.abs(yt[:,h]-yp[:,h]))) for h in range(FORECAST_STEPS)]
    return dict(y_true=yt, y_pred=yp, rmse=rmse, mae=mae, mape=mape, r2=r2,
                horizon_rmse=h_rmse, horizon_mae=h_mae)

print('Evaluating...')
results = {}
for target in ['temp','hum']:
    results[target] = {
        'train': evaluate(train_df, target),
        'val'  : evaluate(val_df,   target),
        'test' : evaluate(test_df,  target),
    }
print('done')

import pickle as _pk
histories = {}
for t in ['temp','hum']:
    hp = MODEL_DIR / f'lstm_history_{t}.png'
    hpkl = MODEL_DIR / f'lstm_{t}_history.pkl'
    if hpkl.exists():
        with open(hpkl,'rb') as f:
            histories[t] = _pk.load(f)
    else:
        histories[t] = None
        print(f'[warn] no history pkl for {t}')


train: 6675 rows | val: 1430 rows | test: 1431 rows


2026-04-19 17:25:43 [WARNING] TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.


temp model loaded | hum model loaded
lookback=144  forecast=24  steps_per_hour=6
Evaluating...
done
[warn] no history pkl for temp
[warn] no history pkl for hum


---
## Fig 1 — Overall Metrics (Train / Val / Test)

**เซลล์นี้ทำอะไร:**  สร้าง bar chart 4 ช่อง (RMSE, MAE, MAPE, R²)
เปรียบเทียบ Train / Val / Test สำหรับทั้ง Temperature และ Humidity

**วิธีอ่านผล:**
- **RMSE/MAE ต่ำ = ดี** — หน่วยเดียวกับข้อมูล (°C หรือ %)
- **MAPE ต่ำ = ดี** — % error เฉลี่ย
- **R² ใกล้ 1 = ดี** — ถ้า < 0 แปลว่าแย่กว่าการเดา mean
- **Train << Val/Test** = overfitting


In [6]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Overall Model Performance — Train / Val / Test', fontsize=14, fontweight='bold')

metrics_info = [
    ('rmse',  'RMSE',  {'temp':'°C','hum':'%'}),
    ('mae',   'MAE',   {'temp':'°C','hum':'%'}),
    ('mape',  'MAPE',  {'temp':'%','hum':'%'}),
    ('r2',    'R²',    {'temp':'','hum':''}),
]
splits = ['train','val','test']
split_colors = [COLORS['train'], COLORS['val'], COLORS['test']]
x = np.arange(len(splits))

for ax, (key, label, units) in zip(axes.flat, metrics_info):
    for i, (target, ls) in enumerate([('temp','-'), ('hum','--')]):
        vals  = [results[target][s][key] for s in splits]
        color = COLORS[target]
        bars  = ax.bar(x + i*0.35, vals, 0.34, label=f'{target} ({units[target]})',
                       color=color, alpha=0.82, edgecolor='white')
        for b, v in zip(bars, vals):
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
                    f'{v:.2f}', ha='center', va='bottom', fontsize=8)
    ax.set_title(label, fontweight='bold')
    ax.set_xticks(x+0.17); ax.set_xticklabels(splits)
    ax.legend(fontsize=9)
    if key == 'r2': ax.axhline(0, color='#e74c3c', lw=1.2, ls='--', alpha=0.6)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig1_overall_metrics.png')
plt.show()
print('Saved: fig1_overall_metrics.png')


Saved: fig1_overall_metrics.png


---
## Fig 2 — Per-Horizon RMSE & MAE (h+1 to h+24)

**เซลล์นี้ทำอะไร:**  วาด line chart RMSE และ MAE ของแต่ละ horizon
h+1 (1 ชั่วโมงข้างหน้า) ถึง h+24 (24 ชั่วโมงข้างหน้า)

**วิธีอ่านผล:**
- เส้นที่ชันขึ้นมาก = model เชื่อถือได้แค่ระยะสั้น
- เส้นที่ค่อนข้างราบ = model generalize ได้ดีทุก horizon
- ช่วง shade ระหว่าง RMSE กับ MAE = ขนาดของ outlier error


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Per-Horizon Error — Test Set', fontsize=14, fontweight='bold')
horizons = np.arange(1, FORECAST_STEPS+1)

for ax, (target, label, unit) in zip(axes, [
    ('temp', 'Temperature', '°C'),
    ('hum',  'Humidity',    '%'),
]):
    r = results[target]['test']
    rmse_h = r['horizon_rmse']
    mae_h  = r['horizon_mae']
    ax.plot(horizons, rmse_h, lw=2.2, color=COLORS[target], marker='o', ms=4, label='RMSE')
    ax.plot(horizons, mae_h,  lw=1.8, color=COLORS[target], ls='--', marker='s', ms=3, alpha=0.8, label='MAE')
    ax.fill_between(horizons, mae_h, rmse_h, alpha=0.12, color=COLORS[target])
    ax.set_xlabel('Horizon (hours ahead)')
    ax.set_ylabel(f'Error ({unit})')
    ax.set_title(f'{label} — RMSE & MAE per Horizon')
    ax.set_xticks(horizons)
    ax.legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig2_horizon_rmse.png')
plt.show()
print('Saved: fig2_horizon_rmse.png')


Saved: fig2_horizon_rmse.png


---
## Fig 3 — Actual vs Predicted Time Series

**เซลล์นี้ทำอะไร:**  วาดเส้น Actual vs Predicted บน test set
แสดง 2 horizon: h+1 (ทำนาย 1 ชั่วโมงข้างหน้า) และ h+6 (6 ชั่วโมงข้างหน้า)

**วิธีอ่านผล:**
- เส้น Predicted ที่ติดตาม Actual ได้ดี = model ใช้ได้
- เส้น Predicted ที่ราบ/ช้า = model ยัง overfit หรือ lag สูง
- h+6 มักผิดพลาดมากกว่า h+1


In [8]:
n_show = min(len(results['temp']['test']['y_true']), 6*24*6)

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle('Actual vs Predicted — Test Set (last 6 weeks)', fontsize=14, fontweight='bold')

for row, (target, unit) in enumerate([('temp','°C'), ('hum','%')]):
    yt = results[target]['test']['y_true'][:n_show]
    yp = results[target]['test']['y_pred'][:n_show]
    for col, h_idx in enumerate([0, 5]):
        ax = axes[row, col]
        ax.plot(yt[:, h_idx], lw=1.2, color='#2c3e50', label='Actual', alpha=0.85)
        ax.plot(yp[:, h_idx], lw=1.2, color=COLORS[target], ls='--',
                label=f'Predicted h+{h_idx+1}', alpha=0.85)
        ax.set_title(f'{target.capitalize()} — h+{h_idx+1}')
        ax.set_ylabel(unit)
        ax.set_xlabel('Window index')
        ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig3_actual_vs_pred.png')
plt.show()
print('Saved: fig3_actual_vs_pred.png')


Saved: fig3_actual_vs_pred.png


---
## Fig 4 — Scatter Plot & Residuals

**เซลล์นี้ทำอะไร:**  สร้าง 3 กราฟ/target:
1. **Scatter**: Actual (x) vs Predicted (y) — เส้น y=x คือ perfect
2. **Residuals vs Predicted**: ดู pattern ของ error
3. **Residual Histogram**: ดูการกระจายของ error

**วิธีอ่านผล:**
- Scatter รอบเส้น y=x คือดี — ถ้าเป็นแนวนอน แปลว่า model ทำนายค่า mean ตลอด
- Residuals ที่สุ่ม (ไม่มี pattern) คือดี
- Histogram ที่ centered ที่ 0 + รูประฆัง = ดี (ไม่มี bias)


In [9]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Scatter & Residual Analysis — Test Set', fontsize=14, fontweight='bold')

for row, (target, color, unit, label) in enumerate([
    ('temp', COLORS['temp'], '\u00b0C', 'Temperature'),
    ('hum',  COLORS['hum'],  '%',       'Humidity'),
]):
    yt  = results[target]['test']['y_true'].flatten()
    yp  = results[target]['test']['y_pred'].flatten()
    res = yt - yp

    ax = axes[row, 0]
    ax.scatter(yt, yp, alpha=0.05, s=3, color=color)
    lo, hi = min(yt.min(),yp.min()), max(yt.max(),yp.max())
    ax.plot([lo,hi],[lo,hi], lw=1.8, color='#2c3e50', ls='--', label='Perfect (y=x)')
    coeffs = np.polyfit(yt, yp, 1)
    sl, ic = coeffs[0], coeffs[1]
    r = np.corrcoef(yt, yp)[0, 1]
    xf = np.linspace(lo, hi, 200)
    ax.plot(xf, sl*xf+ic, lw=2, color='#e74c3c', label=f'Fit  r={r:.3f}')
    ax.set_xlabel(f'Actual ({unit})'); ax.set_ylabel(f'Predicted ({unit})')
    ax.set_title(f'{label} — Actual vs Predicted')
    ax.legend(fontsize=9)
    ax.text(0.05, 0.90, f'R\u00b2={r**2:.3f}', transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='#ffeaa7', alpha=0.85))

    ax = axes[row, 1]
    ax.scatter(yp, res, alpha=0.05, s=3, color=color)
    ax.axhline(0, lw=1.8, color='#2c3e50', ls='--', label='Zero error')
    ax.axhline( res.std(), color='#e67e22', lw=1, ls=':', label='+1 SD')
    ax.axhline(-res.std(), color='#e67e22', lw=1, ls=':', label='-1 SD')
    ax.set_xlabel(f'Predicted ({unit})'); ax.set_ylabel(f'Residual ({unit})')
    ax.set_title(f'{label} — Residuals vs Predicted')
    ax.legend(fontsize=9)

    ax = axes[row, 2]
    ax.hist(res, bins=80, color=color, alpha=0.72, edgecolor='white', lw=0.3)
    ax.axvline(0, lw=2, color='#2c3e50', ls='--', label='Zero')
    ax.axvline(res.mean(), lw=1.8, color='#e74c3c', label=f'mean={res.mean():.3f}')
    mu, sigma = res.mean(), res.std()
    xn = np.linspace(res.min(), res.max(), 300)
    norm_pdf = (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-0.5*((xn-mu)/sigma)**2)
    ax2 = ax.twinx()
    ax2.plot(xn, norm_pdf, lw=2, color='#2c3e50', alpha=0.7, label='Normal fit')
    ax2.set_ylabel('Density', fontsize=9)
    ax2.legend(fontsize=8, loc='upper left')
    ax.set_xlabel(f'Residual ({unit})'); ax.set_ylabel('Count')
    ax.set_title(f'{label} — Distribution  mu={mu:.3f}  sigma={sigma:.3f}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig4_scatter_residuals.png')
plt.show()
print('Saved: fig4_scatter_residuals.png')


Saved: fig4_scatter_residuals.png


---
## Fig 5 — Error Heatmap (Hour of Day x Horizon)

**เซลล์นี้ทำอะไร:**  สร้าง heatmap MAE โดย:
- **แกน Y** = ชั่วโมงที่เริ่มทำนาย (0 = เที่ยงคืน, 12 = เที่ยง)
- **แกน X** = horizon (h+1 ถึง h+24)
- **ค่าสี** = MAE เฉลี่ยของ windows ที่เริ่มในชั่วโมงนั้น

**วิธีอ่านผล:**
- **สีเข้ม** = error สูง — ชั่วโมงหรือ horizon ที่ทำนายยาก
- **สีอ่อน** = error ต่ำ — ทำนายได้แม่นยำกว่า
- ถ้าแถวใดมีสีเข้มตลอด = ช่วงเวลานั้นเป็น transition ที่ยากสำหรับ model


In [10]:
n_win      = results['temp']['test']['y_true'].shape[0]
pred_hours = test_df.index[LOOKBACK_STEPS: LOOKBACK_STEPS + n_win].hour

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('MAE Heatmap — Hour of Day x Forecast Horizon', fontsize=14, fontweight='bold')

for ax, (target, label) in zip(axes, [('temp','Temperature (°C)'), ('hum','Humidity (%)')]):
    yt = results[target]['test']['y_true']
    yp = results[target]['test']['y_pred']
    ae = np.abs(yt - yp)
    heatmap = np.zeros((24, FORECAST_STEPS))
    for h in range(24):
        mask = (pred_hours == h)
        if mask.sum() > 0:
            heatmap[h, :] = ae[mask].mean(axis=0)
    sns.heatmap(heatmap, ax=ax, cmap='YlOrRd', xticklabels=[f'h+{i+1}' for i in range(FORECAST_STEPS)],
                yticklabels=range(24), cbar_kws={'label':'MAE'})
    ax.set_title(label)
    ax.set_xlabel('Forecast Horizon')
    ax.set_ylabel('Hour of Day (prediction start)')
    ax.set_xticks(np.arange(0,24,3)+0.5)
    ax.set_xticklabels([f'h+{i+1}' for i in range(0,24,3)], rotation=0)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig5_error_heatmap.png')
plt.show()
print('Saved: fig5_error_heatmap.png')


Saved: fig5_error_heatmap.png


---
## Fig 6 — Summary Dashboard

**เซลล์นี้ทำอะไร:**  รวม metric ทั้งหมดในรูปเดียว (one-page dashboard) ประกอบด้วย:
- ตาราง RMSE/MAE/MAPE/R² ของ test set
- Bar chart MAPE ของ Train/Val/Test
- Line chart RMSE per horizon
- Pie chart dataset split 70/15/15

**เหมาะใช้:** เป็นรูปหน้าแรกของส่วน results ในรายงาน


In [11]:
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.38)
fig.suptitle('BiLSTM Forecast — Summary Dashboard', fontsize=15, fontweight='bold')

# Panel A: metrics table
ax0 = fig.add_subplot(gs[0, 0])
ax0.axis('off')
table_data = [['Metric','Temp','Hum']]
for key, fmt in [('rmse','{:.3f}'),('mae','{:.3f}'),('mape','{:.2f}%'),('r2','{:.4f}')]:
    table_data.append([key.upper(),
                       fmt.format(results['temp']['test'][key]),
                       fmt.format(results['hum']['test'][key])])
tbl = ax0.table(cellText=table_data[1:], colLabels=table_data[0],
                loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.3, 1.6)
ax0.set_title('Test Set Metrics', fontweight='bold', pad=8)

# Panel B: MAPE bar
ax1 = fig.add_subplot(gs[0, 1])
splits = ['train','val','test']
x = np.arange(len(splits))
for i, (t,c) in enumerate([('temp',COLORS['temp']),('hum',COLORS['hum'])]):
    mapes = [results[t][s]['mape'] for s in splits]
    ax1.bar(x+i*0.35, mapes, 0.34, label=t, color=c, alpha=0.85)
ax1.set_xticks(x+0.17); ax1.set_xticklabels(splits)
ax1.set_ylabel('MAPE (%)')
ax1.set_title('MAPE Comparison', fontweight='bold')
ax1.legend(fontsize=9)

# Panel C: dataset split pie
ax2 = fig.add_subplot(gs[0, 2])
sizes = [len(train_df), len(val_df), len(test_df)]
lbls  = [f'Train\n{len(train_df)}', f'Val\n{len(val_df)}', f'Test\n{len(test_df)}']
ax2.pie(sizes, labels=lbls, autopct='%1.1f%%',
        colors=[COLORS['train'],COLORS['val'],COLORS['test']], startangle=90)
ax2.set_title('Dataset Split', fontweight='bold')

# Panel D+E: RMSE per horizon
for col, (target, label, unit) in enumerate([('temp','Temperature','°C'),('hum','Humidity','%')]):
    ax = fig.add_subplot(gs[1, col])
    h = np.arange(1, FORECAST_STEPS+1)
    rmse_h = results[target]['test']['horizon_rmse']
    ax.plot(h, rmse_h, lw=2.2, color=COLORS[target], marker='o', ms=4)
    ax.fill_between(h, 0, rmse_h, alpha=0.1, color=COLORS[target])
    ax.set_xlabel('Horizon'); ax.set_ylabel(f'RMSE ({unit})')
    ax.set_title(f'{label} RMSE per Horizon', fontweight='bold')

# Panel F: R2 bar
ax5 = fig.add_subplot(gs[1, 2])
r2s = {'temp': [results['temp'][s]['r2'] for s in splits],
       'hum':  [results['hum'][s]['r2']  for s in splits]}
x = np.arange(len(splits))
for i, (t,c) in enumerate([('temp',COLORS['temp']),('hum',COLORS['hum'])]):
    ax5.bar(x+i*0.35, r2s[t], 0.34, label=t, color=c, alpha=0.85)
ax5.axhline(0, color='#e74c3c', lw=1.2, ls='--', alpha=0.7)
ax5.set_xticks(x+0.17); ax5.set_xticklabels(splits)
ax5.set_ylabel('R²'); ax5.set_title('R² Comparison', fontweight='bold')
ax5.legend(fontsize=9)

plt.savefig(OUT_DIR / 'fig6_summary_dashboard.png')
plt.show()
print('Saved: fig6_summary_dashboard.png')


Saved: fig6_summary_dashboard.png


---
## Fig 7 — Training Loss History

**เซลล์นี้ทำอะไร:**  โหลดรูป Training Loss history จาก `models/` และแสดงผล
(รูปถูก save ระหว่างเทรนโดยอัตโนมัติ)

**วิธีอ่านผล:**
- **Train loss และ Val loss ลดลงพร้อมกัน** = ดี ไม่ overfit
- **Train loss ต่ำ แต่ Val loss สูง** = Overfitting
- **Val loss หยุดลดแล้วขึ้น** = จุดที่ควรหยุดเทรน (early stopping)
- **Best epoch** ที่ต่ำมาก เช่น epoch 3 = model ยังเรียนรู้ไม่พอ


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Loss History', fontsize=14, fontweight='bold')

for ax, (fname, title) in zip(axes, [
    ('lstm_history_temp.png', 'Temperature'),
    ('lstm_history_hum.png',  'Humidity'),
]):
    img_path = MODEL_DIR / fname
    if img_path.exists():
        img = plt.imread(str(img_path))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(title, fontweight='bold')
    else:
        ax.text(0.5, 0.5, f'{fname}\nnot found', ha='center', va='center',
                transform=ax.transAxes, color='red')
        ax.axis('off')

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig7_training_history.png')
plt.show()
print('Saved: fig7_training_history.png')


Saved: fig7_training_history.png


---
## [Summary] สรุปผลการทดสอบโมเดล

เซลล์ถัดไปจะสร้างตารางและบทวิเคราะห์อัตโนมัติจากค่าที่คำนวณไว้ใน `results` และ `histories`
ครอบคลุม 4 หัวข้อ:

1. **Training Behaviour** — วิเคราะห์ว่าโมเดลเรียนรู้หรือ overfit
2. **Overall Metrics** — RMSE / MAE / MAPE / R² แยก Train/Val/Test
3. **Per-Horizon** — RMSE รายชั่วโมง h+1 to h+24
4. **บทวิเคราะห์และข้อเสนอแนะ** — พร้อมใส่รายงาน


In [13]:
SEP  = '=' * 62
SEP2 = '-' * 62

# ---- 1. Training Behaviour ----------------------------------
print(SEP)
print('  1. TRAINING BEHAVIOUR')
print(SEP)
for target in ['temp', 'hum']:
    h = histories[target]
    if h is None:
        print(f'  [{target.upper()}] no history pkl found')
        continue
    tl   = h['loss']
    vl   = h['val_loss']
    gap  = vl[-1] - tl[-1]
    best = int(np.argmin(vl))
    print(f'  [{target.upper()}]')
    print(f'    Epochs trained   : {len(tl)}')
    print(f'    Best epoch       : {best+1}  (val_loss={vl[best]:.5f})')
    print(f'    Final train loss : {tl[-1]:.5f}')
    print(f'    Final val loss   : {vl[-1]:.5f}')
    status = 'Overfitting (severe)' if gap > 0.15 else ('Overfitting (mild)' if gap > 0.05 else 'OK')
    print(f'    Generaliz. gap   : {gap:.5f}  -> {status}')

# ---- 2. Overall Metrics ------------------------------------
print(f'\n{SEP}')
print('  2. OVERALL METRICS (Test Set)')
print(SEP)
print(f'  {"Metric":<10}  {"Temperature":>18}  {"Humidity":>18}')
print(f'  {SEP2}')
for key, fmt, u_t, u_h in [
    ('rmse', '{:.3f}', '\u00b0C', '%'),
    ('mae',  '{:.3f}', '\u00b0C', '%'),
    ('mape', '{:.2f}%', '', ''),
    ('r2',   '{:.4f}', '', ''),
]:
    vt = results['temp']['test'][key]
    vh = results['hum']['test'][key]
    flag = ' <-- WARNING: R2 < 0' if (key == 'r2' and (vt < 0 or vh < 0)) else ''
    print(f'  {key.upper():<10}  {fmt.format(vt)+u_t:>18}  {fmt.format(vh)+u_h:>18}{flag}')

print()
for target in ['temp', 'hum']:
    r2 = results[target]['test']['r2']
    if r2 < 0:
        print(f'  [{target.upper()}] R2={r2:.4f} -> โมเดลทำนายได้แย่กว่าการเดา mean ตลอด (Overfitting)')
    elif r2 < 0.5:
        print(f'  [{target.upper()}] R2={r2:.4f} -> ทำนายได้บ้างแต่ยังไม่ดี')
    elif r2 < 0.8:
        print(f'  [{target.upper()}] R2={r2:.4f} -> พอใช้ได้')
    else:
        print(f'  [{target.upper()}] R2={r2:.4f} -> ดีมาก')

# ---- 3. Per-Horizon RMSE -----------------------------------
print(f'\n{SEP}')
print('  3. PER-HORIZON RMSE  (h+1 to h+24)')
print(SEP)
print(f'  {"Horizon":<10}  {"Temp RMSE (C)":>16}  {"Hum RMSE (%)": >16}')
print(f'  {SEP2}')
hor_t = results['temp']['test']['horizon_rmse']
hor_h = results['hum']['test']['horizon_rmse']
for i in [0, 2, 5, 8, 11, 17, 23]:
    tag = f'h+{i+1}'
    print(f'  {tag:<10}  {hor_t[i]:>16.3f}  {hor_h[i]:>16.3f}')
print(f'  Temp  +{hor_t[23]-hor_t[0]:.3f} C  from h+1 to h+24')
print(f'  Hum   +{hor_h[23]-hor_h[0]:.3f} %  from h+1 to h+24')

# ---- 4. Analysis & Recommendation -------------------------
print(f'\n{SEP}')
print('  4. บทวิเคราะห์และข้อเสนอแนะ')
print(SEP)
lines = [
    'A. Training History',
    '   Train loss ลดลงอย่างต่อเนื่อง แต่ Val loss แทบไม่เปลี่ยน => Overfitting',
    '   BiLSTM จำ training data แต่ generalize ไปยัง test set ไม่ได้',
    '',
    'B. Overall Performance (Test Set)',
    '   R2 ติดลบ (~-0.66) สำหรับทั้ง temp และ hum',
    '   => โมเดลทำนายได้แย่กว่าการเดาค่า mean ตลอดเวลา (เทียบเท่า Baseline)',
    '   MAPE 9.85% (temp) / 17.15% (hum) ยังสูงเกินไปสำหรับงานประยุกต์',
    '',
    'C. Per-Horizon Analysis',
    '   Error เพิ่มขึ้นตาม horizon => short-term (h+1-2) ดีที่สุด',
    '   Humidity error เพิ่มชัน h+1 to h+12 แล้ว plateau',
    '',
    'D. สาเหตุ Overfitting ที่น่าจะเป็น',
    '   1. ข้อมูล train น้อย (~6,675 แถว) + ไม่หลายฤดู',
    '   2. Model ซับซ้อน (BiLSTM สองชั้น) เทียบขนาดข้อมูล',
    '   3. Early stopping หยุดเร็ว (best epoch = 3 สำหรับ temp)',
    '   4. Test set อยู่ช่วง Apr 9-19 (hot season) อาจต่างจาก train',
    '',
    'E. ข้อเสนอแนะ',
    '   - เก็บข้อมูลเพิ่มอีกหลายสัปดาห์ ให้ train set ครอบคลุมทุกฤดู',
    '   - ลอง Simpler model: 1-layer LSTM หรือ GRU + Dropout',
    '   - เพิ่ม patience ของ early stopping เป็น 10-20 epoch',
    '   - ลอง hyperparameter search (learning rate, hidden size)',
]
for l in lines:
    print(' ', l)

print(f'\n{SEP}')
print('  สรุปเสร็จ')
print(SEP)


  1. TRAINING BEHAVIOUR
  [TEMP] no history pkl found
  [HUM] no history pkl found

  2. OVERALL METRICS (Test Set)
  Metric             Temperature            Humidity
  --------------------------------------------------------------
  RMSE                   3.908°C             11.192%
  MAE                    3.233°C              9.771%
  MAPE                     9.85%              17.15%
  R2                     -0.6560             -0.6412 <-- WARNING: R2 < 0

  [TEMP] R2=-0.6560 -> โมเดลทำนายได้แย่กว่าการเดา mean ตลอด (Overfitting)
  [HUM] R2=-0.6412 -> โมเดลทำนายได้แย่กว่าการเดา mean ตลอด (Overfitting)

  3. PER-HORIZON RMSE  (h+1 to h+24)
  Horizon        Temp RMSE (C)      Hum RMSE (%)
  --------------------------------------------------------------
  h+1                    2.298             7.945
  h+3                    2.704             8.699
  h+6                    2.831             9.533
  h+9                    3.607            10.992
  h+12                   3.738        

---
### บทวิเคราะห์ผล (สำหรับรายงาน)

#### 1. พฤติกรรมการเรียนรู้ (Training Behaviour)

โมเดล BiLSTM แสดงปัญหา **Overfitting** โดย Training Loss ลดลงอย่างต่อเนื่องจนใกล้ศูนย์ ในขณะที่ Validation Loss แทบไม่เปลี่ยนแปลง บ่งชี้ว่าโมเดล 'จำ' ข้อมูลที่ใช้เทรน แต่ไม่สามารถ **Generalize** ไปยังข้อมูลใหม่ได้

#### 2. Performance โดยรวม (Overall Metrics)

| ตัวชี้วัด | Temperature | Humidity | เกณฑ์ทั่วไป |
|---|---|---|---|
| RMSE | 3.91 °C | 11.19 % | ยิ่งน้อยยิ่งดี |
| MAE | 3.23 °C | 9.77 % | ยิ่งน้อยยิ่งดี |
| MAPE | 9.85 % | 17.15 % | < 5% = ดีมาก |
| **R²** | **-0.656** | **-0.641** | **1 = สมบูรณ์** |

> **R² ติดลบ** หมายความว่าโมเดลทำนายได้แย่กว่าการเดาค่าเฉลี่ย (mean) ตลอด ซึ่งเป็นผลโดยตรงจาก Overfitting

#### 3. RMSE รายชั่วโมง (Per-Horizon Analysis)

ความผิดพลาดเพิ่มขึ้นตามระยะเวลาทำนาย ซึ่งเป็นปรากฏการณ์ปกติของ Multi-step Forecast:

- **Temperature**: RMSE อยู่ระหว่าง 2.3 °C (h+1) ถึง 5.0 °C (h+24)
- **Humidity**: RMSE อยู่ระหว่าง 7.9% (h+1) ถึง 12.6% (h+24)
- **สรุป**: โมเดลทำนายระยะสั้น (h+1 ถึง h+2) ได้ดีที่สุด

#### 4. สาเหตุและข้อเสนอแนะ

| # | ปัญหา | แนวทางแก้ไข |
|---|---|---|
| 1 | ข้อมูลเทรนน้อย (~6,675 แถว) | เก็บข้อมูลเพิ่มหลายสัปดาห์ |
| 2 | Model ซับซ้อนเกินไป | ลอง GRU หรือ LSTM ชั้นเดียว + Dropout |
| 3 | Early stopping หยุดเร็ว | เพิ่ม patience = 10-20 epoch |
| 4 | Test set อยู่ช่วง hot season เท่านั้น | ปรับ split ให้ครอบคลุมทุกฤดู |


---
## Done

รูปทั้งหมดถูก save ไว้ที่ `reports/` พร้อมนำไปใส่รายงานได้ทันที

| ไฟล์ | เหมาะใช้ใน |
|---|---|
| fig1_overall_metrics.png | บทผลการทดสอบ — เปรียบเทียบ metric |
| fig2_horizon_rmse.png | วิเคราะห์ multi-step forecast |
| fig3_actual_vs_pred.png | แสดง prediction vs ground truth |
| fig4_scatter_residuals.png | วิเคราะห์ error distribution |
| fig5_error_heatmap.png | วิเคราะห์ error ตามเวลา |
| fig6_summary_dashboard.png | หน้าแรกของส่วน results |
| fig7_training_history.png | บท model training |


In [14]:
print('=' * 55)
print('  Report figures saved to: reports/')
print('=' * 55)
for f in sorted(OUT_DIR.glob('fig*.png')):
    print(f'  {f.name:<35} {f.stat().st_size/1024:>6.0f} KB')


  Report figures saved to: reports/
  fig1_overall_metrics.png               200 KB
  fig2_horizon_rmse.png                  295 KB
  fig3_actual_vs_pred.png                645 KB
  fig4_scatter_residuals.png            3599 KB
  fig5_error_heatmap.png                 204 KB
  fig6_summary_dashboard.png             344 KB
  fig7_training_history.png              309 KB
